# config

In [1]:
config = {
    "file_config": {
        "input_path": "",
        "output_path": ""
        
    },
    "custom_config": {
        "spark_profile": "test",  # 测试改 test，全量改 prod
        "input_summary_path": "",
    }
}


In [ ]:
from __future__ import annotations
import os
os.environ["SPARK_USER"] = "renpengli"
os.environ.setdefault("XINGHE_CONF_DIR", "/share/renpengli/conf")

import copy
import json
import math
import socket
import time
import logging
import threading
import traceback
import uuid
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Any
import httpx
from boto3.s3.transfer import TransferConfig
from dotenv import load_dotenv
from bisect import bisect_left
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone

from pyspark import SparkConf, StorageLevel
from pyspark.sql import SparkSession, Row
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.functions import col, from_json, lower, transform, udf
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, MapType, ArrayType, LongType

from xinghe.s3 import *
from xinghe.utils.json_util import *
# from xinghe.s3 import delete_s3_objects, list_s3_objects, put_s3_object, read_s3_object_bytes
# from xinghe.s3.path import split_s3_path
import xinghe.s3.upload as xinghe_s3_upload
import xinghe.s3.write as xinghe_s3_write
import xinghe.s3.client as xinghe_s3_client
import xinghe.s3.retry as xinghe_s3_retry

from admin_ingest.settings import Settings
from admin_ingest.spark_bridge import embed_record_texts

ENV_PATH = "/share/renpengli/conf/agent_search/.env"
if os.path.exists(ENV_PATH):
    load_dotenv(ENV_PATH, override=True)
else:
    from admin_ingest.env_bootstrap import load_dotenvs
    load_dotenvs()
    
settings = Settings.load()

# config = json_loads('${config}')
custom_config = config["custom_config"]
file_config = config["file_config"]
s3_config = config.get("s3_config", {})

spark_profile = custom_config["spark_profile"]

def get_s3_args(path, s3_config):
    bucket = split_s3_path(path)[0]
    s3_args = s3_config.get(bucket, {})
    if not s3_args:
        s3_args = get_s3_config(path)
    print(f"bucket={bucket}, s3_args={s3_args}")
    return bucket, s3_args

INPUT_ROOT = file_config["input_path"]
read_s3_bucket, read_s3_config = get_s3_args(INPUT_ROOT, s3_config)

INPUT_SUMMARY_PATH = custom_config["input_summary_path"]
OUTPUT_ROOT = file_config["output_path"].rstrip('/')
PROGRESS_ROOT = f"{OUTPUT_ROOT.rstrip('/')}/_progress"
SUMMARY_PATH = f"{OUTPUT_ROOT.rstrip('/')}/_summary.json"
ERROR_ROOT = f"{OUTPUT_ROOT.rstrip('/')}/error"
OUTPUT_WRITE_MODE = custom_config.get("output_write_mode", "overwrite")
OVERWRITE_OUTPUT = custom_config.get("overwrite_output", False)
DELETE_BATCH_OUTPUT_BEFORE_WRITE = custom_config.get("delete_batch_output_before_write", OUTPUT_WRITE_MODE != "append")
TEST_OUTPUT_ROOT = OUTPUT_ROOT
TEST_PROGRESS_ROOT = f"{TEST_OUTPUT_ROOT.rstrip('/')}/_progress"
TEST_SUMMARY_PATH = f"{TEST_OUTPUT_ROOT.rstrip('/')}/_summary.json"
TEST_ERROR_ROOT = f"{TEST_OUTPUT_ROOT.rstrip('/')}/error"
MILVUS_FIELDS = [
    "chunk_id",
    "doc_id",
    "title",
    "url",
    "embedding",
    "offset",
    "page_no",
    "chunk_no",
    "source_type",
    "lang",
    "category",
    "job_id",
    "task_id",
    "publication_published_date",
    "publication_published_year",
    "metadata_type",
    "publication_venue_type",
    "primary_topic",
    "primary_topic_subfield",
    "primary_topic_field",
    "primary_topic_domain",
    "author",
    "publication_venue_name_unified",
    "access_is_oa",
    "citation_count",
    "influential_citation_count",
    "extra_info",
    "created_time",
    "updated_time",
]
SCALAR_INDEX_FIELDS = [
    "title",
       "metadata_type",
    "publication_venue_type",
     "primary_topic_field",
    "primary_topic_domain",
    "publication_venue_name_unified",
    "access_is_oa",
    "author",
    "publication_published_date",
    "publication_published_year",
    "citation_count",
    "influential_citation_count",
]
assert len(MILVUS_FIELDS) == 29
assert set(SCALAR_INDEX_FIELDS).issubset(set(MILVUS_FIELDS))
PROMOTED_EXTRA_INFO_KEYS = {
    "record_type",
    "chunk_id",
    "doc_id",
    "title",
    "content",
    "url",
    "category",
    "source_type",
    "lang",
    "job_id",
    "task_id",
    "offset",
    "page_no",
    "chunk_no",
    "created_time",
    "updated_time",
    "publication_published_date",
    "publication_published_year",
    "metadata_type",
    "publication_venue_type",
    "primary_topic",
    "primary_topic_subfield",
    "primary_topic_field",
    "primary_topic_domain",
    "author",
    "publication_venue_name_unified",
    "access_is_oa",
    "citation_count",
    "influential_citation_count",
}
MAX_LEN = {
    "chunk_id": 512,
    "doc_id": 512,
    "title": 4096,
    "url": 4096,
    "source_type": 128,
    "lang": 64,
    "category": 512,
    "job_id": 256,
    "task_id": 256,
    "metadata_type": 32,
    "publication_venue_type": 64,
    "primary_topic": 512,
    "primary_topic_subfield": 512,
    "primary_topic_field": 512,
    "primary_topic_domain": 512,
    "publication_venue_name_unified": 512,
    "access_is_oa": 32,
    "extra_info": 32768,
    "author_item": 512,
}
AUTHOR_MAX_CAPACITY = 64
AUTHOR_DEBUG_MAX_LOGS = 20
AUTHOR_DEBUG_LOG_COUNT = 0
RESULT_ROW_SCHEMA = T.StructType([
    T.StructField("value", T.StringType(), False),
    T.StructField("sub_path", T.StringType(), False),
    T.StructField("metric_calls", T.LongType(), False),
    T.StructField("metric_texts", T.LongType(), False),
    T.StructField("metric_seconds", T.DoubleType(), False),
])
PROGRESS_SCHEMA = T.StructType([
    T.StructField("batch_index", T.LongType(), False),
    T.StructField("batch_start_row", T.LongType(), False),
    T.StructField("batch_end_row", T.LongType(), False),
    T.StructField("completed_row_count", T.LongType(), False),
    T.StructField("last_completed_source_file", T.StringType(), True),
    T.StructField("last_completed_chunk_id", T.StringType(), True),
    T.StructField("success_row_count", T.LongType(), False),
    T.StructField("error_row_count", T.LongType(), False),
    T.StructField("completed_at", T.StringType(), False),
])

INPUT_FILE_SCHEMA = T.StructType([
    T.StructField("record_type", T.StringType(), True),
    T.StructField("chunk_id", T.StringType(), True),
    T.StructField("doc_id", T.StringType(), True),
    T.StructField("title", T.StringType(), True),
    T.StructField("content", T.StringType(), True),
    T.StructField("url", T.StringType(), True),
    T.StructField("category", T.StringType(), True),
    T.StructField("source_type", T.StringType(), True),
    T.StructField("lang", T.StringType(), True),
    T.StructField("job_id", T.StringType(), True),
    T.StructField("task_id", T.StringType(), True),
    T.StructField("extra_info", T.StringType(), True),
    T.StructField("created_time", T.StringType(), True),
    T.StructField("updated_time", T.StringType(), True),
    T.StructField("offset", T.LongType(), True),
    T.StructField("page_no", T.LongType(), True),
    T.StructField("chunk_no", T.LongType(), True),
    T.StructField("publication_published_date", T.StringType(), True),
    T.StructField("publication_published_year", T.StringType(), True),
    T.StructField("metadata_type", T.StringType(), True),
    T.StructField("publication_venue_type", T.StringType(), True),
    T.StructField("primary_topic", T.StringType(), True),
    T.StructField("primary_topic_subfield", T.StringType(), True),
    T.StructField("primary_topic_field", T.StringType(), True),
    T.StructField("primary_topic_domain", T.StringType(), True),
    T.StructField("author", T.StringType(), True),
    T.StructField("publication_venue_name_unified", T.StringType(), True),
    T.StructField("access_is_oa", T.StringType(), True),
    T.StructField("citation_count", T.StringType(), True),
    T.StructField("influential_citation_count", T.StringType(), True),
])


bucket=web-parse-hw60p, s3_args={'endpoint': 'http://d-ceph-ssd-inside.pjlab.org.cn', 'ak': 'C5A40882720307B005DC', 'sk': 'p2uzbWzjLLNmo5yNZeYzUDIKLQAAAAGTcgMHvbRb'}


# OA chunk embedding 生产 notebook

目标：
- 输入 `s3://web-parse-hw60p/linfeng/agentic_search/oa_chunk_data/v2/data/`
- 只复用 `embed_record_texts(...)` 的调用方式
- 输出字段为 `es-milvus-全量入库字段与索引表.md` 中 Milvus 主集合 29 个字段
- 用 `xinghe` 读写 S3，按批次写出
- 支持 20 条测试与断点续跑


In [3]:
if spark_profile == "prod":
    base_config = {
        # "spark.dynamicAllocation.minExecutors": "20",
        # "spark.dynamicAllocation.initialExecutors": "40",
        # "spark.dynamicAllocation.maxExecutors": "200",
        "spark.default.parallelism": "1000",
        "spark.sql.shuffle.partitions": "1000",
        "spark.executor.memory": "8g",
        "spark.executor.memoryOverhead": "3g",
        "spark.driver.memory": "8g",
        "spark.driver.maxResultSize": "4g",
        "spark.executor.cores": "4",
        "spark.executor.instances": "200",
    }
else:
    base_config = {
        # "spark.dynamicAllocation.minExecutors": "2",
        # "spark.dynamicAllocation.initialExecutors": "4",
        # "spark.dynamicAllocation.maxExecutors": "16",
        "spark.default.parallelism": "64",
        "spark.sql.shuffle.partitions": "64",
        "spark.executor.memory": "4g",
        "spark.executor.memoryOverhead": "2g",
        "spark.driver.memory": "4g",
        "spark.driver.maxResultSize": "2g",
        "spark.executor.cores": "4",
        "spark.executor.instances": "20",
    }
    
spark_config = {
    **base_config,
    # spark 资源配置
    "spark.kubernetes.executor.limit.cores": "8",  # spark.kubernetes.executor.limit.cores 是对cpu的限制
    # spark 其他配置
    "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
    "spark.speculation": "true",
    "spark.executorEnv.SPARK_USER": "renpengli",
    #sparksql 配置
    "spark.sql.sources.partitionOverwriteMode":"dynamic",
    "spark.sql.parquet.compression.codec":"snappy",
    "spark.sql.files.maxRecordsPerFile": 50000,
    "spark.sql.adaptive.enabled":"true",
    "spark.sql.adaptive.coalescePartitions.enabled":"true",
    "spark.sql.adaptive.advisoryPartitionSizeInBytes":"256MB",
    "spark.sql.autoBroadcastJoinThreshold":"-1",
    "spark.sql.adaptive.shuffle.targetPostShuffleInputSize":"67108864",
    # hadoop s3a 配置
    "spark.hadoop.fs.s3a.impl":"org.apache.hadoop.fs.s3a.S3AFileSystem",
    "spark.hadoop.fs.s3a.connection.ssl.enabled": "false",
    "spark.hadoop.fs.s3a.path.style.access": "true",
    "spark.hadoop.fs.s3a.endpoint": read_s3_config["endpoint"],
    "spark.hadoop.fs.s3a.access.key": read_s3_config["ak"],
    "spark.hadoop.fs.s3a.secret.key": read_s3_config["sk"],
    "spark.hadoop.fs.s3a.multiobjectdelete.enable":"false",
    "spark.hadoop.fs.s3a.directory.marker.retention":"keep",
    "spark.hadoop.fs.s3a.fast.upload":"true",
    "spark.hadoop.fs.s3a.connection.maximum":"1000",
    "spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version":"2",
    # kubernetes 配置
    "spark.kubernetes.executor.deleteOnTermination":"false",
    "spark.kubernetes.namespace": "dataops-paas",
    "spark.kubernetes.authenticate.driver.serviceAccountName": "spark-default",
    "spark.kubernetes.container.image.pullPolicy": "Always",
    "spark.kubernetes.container.image": "registry.sensetime.com/hadoop/dataops/apache/spark:3.5.7-data-platform",
    # event log 配置
    "spark.eventLog.enabled": "true",
    "spark.eventLog.dir": "/spark/eventLog",
    # Magic Committer（推荐 - 不需要临时目录，性能最佳）
    # 参考: https://hadoop.apache.org/docs/current/hadoop-aws/tools/hadoop-aws/committers.html
    "spark.jars":"/share/spark-jars/spark-hadoop-cloud_2.12-3.5.7.jar",
    "spark.hadoop.fs.s3a.committer.name": "magic",
    "spark.hadoop.fs.s3a.committer.magic.enabled": "true" ,
    "spark.hadoop.mapreduce.outputcommitter.factory.scheme.s3a": "org.apache.hadoop.fs.s3a.commit.S3ACommitterFactory",
    "spark.sql.sources.commitProtocolClass": "org.apache.spark.internal.io.cloud.PathOutputCommitProtocol",
    "spark.sql.parquet.output.committer.class": "org.apache.spark.internal.io.cloud.BindingParquetOutputCommitter",
    "spark.kubernetes.executor.podTemplateFile": "/share/renpengli/pod-template/pod-template-dolphin.yaml",
    "spark.kubernetes.executor.label.queue": "root.datacenter.data-producer.default",
    "spark.driver.host": os.environ.get("POD_IP"),
    "spark.submit.deployMode": "client",
}

conf = SparkConf() 
conf.setAll(list(spark_config.items()))

# 初始化Spark
master = "k8s://https://{kubernetes_service_host}:{kubernetes_service_port}".format(kubernetes_service_host = os.environ.get("KUBERNETES_SERVICE_HOST"), 
        kubernetes_service_port = os.environ.get("KUBERNETES_SERVICE_PORT"))
tt = int(time.time())
spark = SparkSession.builder.master(master).config(conf=conf).appName(f"chunk_to_embedding_{tt}").getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")
spark


26/06/25 15:50:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/25 15:50:31 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/25 15:50:31 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/06/25 15:50:31 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


# funcs

In [4]:
def patch_xinghe_s3_multipart_checksum(checksum_algorithm: str = "SHA256", multipart_threshold_mb: int = 128, multipart_chunksize_mb: int = 16, max_retries: int = 5, retry_sleep_seconds: int = 180) -> None:
    multipart_threshold = int(multipart_threshold_mb) * 1024 ** 2
    multipart_chunksize = int(multipart_chunksize_mb) * 1024 ** 2
    def upload_s3_object_with_checksum(path: str, local_file_path: str, client=None):
        client = client or xinghe_s3_client.get_s3_client(path)
        config = TransferConfig(
            multipart_threshold=multipart_threshold,
            multipart_chunksize=multipart_chunksize,
        )
        bucket, key = split_s3_path(path)
        extra_args = None
        try:
            if os.path.getsize(local_file_path) >= multipart_threshold:
                extra_args = {"ChecksumAlgorithm": checksum_algorithm}
        except OSError:
            extra_args = {"ChecksumAlgorithm": checksum_algorithm}
        if extra_args:
            client.upload_file(local_file_path, bucket, key, ExtraArgs=extra_args, Config=config)
        else:
            client.upload_file(local_file_path, bucket, key, Config=config)
    def upload_s3_object_with_checksum_retry(path: str, local_file_path: str, client=None):
        last_exc = None
        for attempt in range(max_retries + 1):
            try:
                return upload_s3_object_with_checksum(path, local_file_path, client=client)
            except Exception as exc:
                last_exc = exc
                if attempt >= max_retries:
                    raise Exception(f"Retry exhausted for [patched_upload_s3_object_with_checksum_retry], args={(path, local_file_path)}") from last_exc
                try:
                    time.sleep(retry_sleep_seconds)
                except Exception:
                    pass
    xinghe_s3_client.upload_s3_object = upload_s3_object_with_checksum
    xinghe_s3_retry.upload_s3_object = upload_s3_object_with_checksum
    xinghe_s3_retry.upload_s3_object_with_retry = upload_s3_object_with_checksum_retry
    xinghe_s3_upload.upload_s3_object_with_retry = upload_s3_object_with_checksum_retry
    xinghe_s3_write.upload_s3_object_with_retry = upload_s3_object_with_checksum_retry
    print(f"patched xinghe s3 multipart upload checksum={checksum_algorithm}", flush=True)
    
patch_xinghe_s3_multipart_checksum()


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="milliseconds").replace("+00:00", "Z")
def log_info(message: str) -> None:
    print(message, flush=True)
def elapsed_seconds(started_at: float) -> float:
    return time.perf_counter() - started_at
def build_debug_item_payload(row: dict[str, Any], content_preview_bytes: int = 2048) -> dict[str, Any]:
    content = first_text(row.get("content"))
    return {
        "chunk_id": first_text(row.get("chunk_id")),
        "doc_id": first_text(row.get("doc_id")),
        "row_loc": first_text(row.get("__row_loc"), row.get("chunk_id")),
        "source_file": first_text(row.get("__source_file")),
        "content_chars": len(content),
        "content_bytes": len(content.encode("utf-8")) if content else 0,
        "content_preview": truncate_utf8(content, content_preview_bytes),
    }
def extract_embedding_error_response(exc: Exception, max_bytes: int = 8192) -> dict[str, Any]:
    payload = {
        "exception_type": exc.__class__.__name__,
        "error_message": str(exc),
    }
    if isinstance(exc, httpx.HTTPStatusError) and exc.response is not None:
        payload["response_status_code"] = exc.response.status_code
        try:
            payload["response_text"] = truncate_utf8(exc.response.text, max_bytes)
        except Exception:
            payload["response_text"] = ""
    return payload
def is_context_length_error(error_payload: dict[str, Any]) -> bool:
    response_text = first_text(error_payload.get("response_text")).lower()
    return "maximum context length" in response_text or "parameter=input_tokens" in response_text or '"param":"input_tokens"' in response_text
def build_embed_batch_error_message(exc: Exception, batch_index: int, items: list[dict[str, Any]], failed_items: list[dict[str, Any]] | None = None) -> str:
    payload = {
        "record_type": "embed_batch_error_debug",
        "batch_index": int(batch_index),
        "buffer_size": len(items),
        "batch_error": extract_embedding_error_response(exc),
    }
    if failed_items is None:
        payload["items"] = [build_debug_item_payload(item) for item in items]
    else:
        payload["failed_items"] = failed_items
        payload["failed_item_count"] = len(failed_items)
        payload["single_item_probe_count"] = len(items)
        if not failed_items:
            payload["single_item_probe_result"] = "all_single_item_requests_succeeded"
    return truncate_utf8(json.dumps(payload, ensure_ascii=False), 131072)
def truncate_utf8(value: Any, max_bytes: int) -> str:
    text = str(value or "").strip()
    raw = text.encode("utf-8")
    if len(raw) <= max_bytes:
        return text
    raw = raw[:max_bytes]
    while raw:
        try:
            return raw.decode("utf-8").rstrip()
        except UnicodeDecodeError:
            raw = raw[:-1]
    return ""
def first_text(*values: Any) -> str:
    for value in values:
        if value is None:
            continue
        text = value.strip() if isinstance(value, str) else str(value).strip()
        if text and text.lower() not in {"null", "none", "nan"}:
            return text
    return ""
def log_author_debug(raw_value: Any, item: Any, context: dict[str, Any] | None = None, reason: str = "") -> None:
    global AUTHOR_DEBUG_LOG_COUNT
    if AUTHOR_DEBUG_LOG_COUNT >= AUTHOR_DEBUG_MAX_LOGS:
        return
    AUTHOR_DEBUG_LOG_COUNT += 1
    payload = {
        "record_type": "author_debug",
        "reason": reason,
        "context": context or {},
        "raw_author_value": raw_value,
        "author_item": item.asDict(recursive=True) if hasattr(item, 'asDict') else item,
        "raw_author_value_type": type(raw_value).__name__,
        "author_item_type": type(item).__name__,
    }
    print("\n===== author_debug =====", flush=True)
    print(json.dumps(payload, ensure_ascii=False, indent=2, default=str), flush=True)
    print("===== /author_debug =====\n", flush=True)
def parse_extra_info(value: Any) -> dict[str, Any]:
    if isinstance(value, dict):
        return copy.deepcopy(value)
    if isinstance(value, str) and value.strip():
        try:
            parsed = json.loads(value)
        except Exception:
            return {}
        return parsed if isinstance(parsed, dict) else {}
    return {}
def parse_topic_dict(value: Any) -> dict[str, Any]:
    if isinstance(value, dict):
        return value
    if isinstance(value, str) and value.strip().startswith("{"):
        try:
            parsed = json.loads(value)
        except Exception:
            return {}
        return parsed if isinstance(parsed, dict) else {}
    return {}
def normalize_bool_string(value: Any) -> str:
    text = first_text(value).lower()
    if text in {"1", "true", "yes", "y"}:
        return "true"
    if text in {"0", "false", "no", "n"}:
        return "false"
    return text
def normalize_int(value: Any, default: int = 0) -> int:
    if value is None:
        return default
    try:
        if isinstance(value, bool):
            return int(value)
        text = str(value).strip()
        if not text or text.lower() in {"null", "none", "nan"}:
            return default
        return int(float(text))
    except Exception:
        return default
def normalize_published_date(value: Any) -> int:
    text = first_text(value)
    if not text:
        return 0
    if len(text) >= 10 and text[4:5] == "-" and text[7:8] == "-":
        text = text[:10]
    for fmt in ("%Y-%m-%d", "%Y-%m", "%Y"):
        try:
            dt = datetime.strptime(text, fmt)
            if fmt == "%Y-%m":
                return int(dt.strftime("%Y%m01"))
            if fmt == "%Y":
                return int(dt.strftime("%Y0101"))
            return int(dt.strftime("%Y%m%d"))
        except Exception:
            pass
    digits = "".join(ch for ch in text if ch.isdigit())
    if len(digits) >= 8:
        return normalize_int(digits[:8], 0)
    return 0
def normalize_ts(value: Any, fallback: str | None = None) -> str:
    text = first_text(value)
    if not text:
        return fallback or utc_now_iso()
    if text.isdigit():
        num = int(text)
        if num < 10_000_000_000:
            num *= 1000
        return datetime.fromtimestamp(num / 1000.0, tz=timezone.utc).isoformat(timespec="milliseconds").replace("+00:00", "Z")
    try:
        dt = datetime.fromisoformat(text.replace("Z", "+00:00"))
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        return dt.astimezone(timezone.utc).isoformat(timespec="milliseconds").replace("+00:00", "Z")
    except Exception:
        return fallback or utc_now_iso()

def normalize_author_list(value: Any, lowercase: bool = True, debug_context: dict[str, Any] | None = None) -> list[str]:
    if value is None:
        return []
    if isinstance(value, list):
        raw_items = value
    else:
        text = first_text(value)
        if text.startswith("[") and text.endswith("]"):
            try:
                parsed = json.loads(text)
                raw_items = parsed if isinstance(parsed, list) else [text]
            except Exception:
                raw_items = [text]
        elif text:
            raw_items = [text]
        else:
            raw_items = []
    raw_items = [first_text(item.get("name")) if isinstance(item, dict) else item for item in raw_items]
    out = []
    for item in raw_items:
        if hasattr(item, 'asDict'):
            item = item.asDict(recursive=True)
        if isinstance(item, dict):
            log_author_debug(value, item, debug_context, reason='dict_reached_lower')
            raise AttributeError("'dict' object has no attribute 'lower'")
        text = truncate_utf8(item.lower() if lowercase else item, MAX_LEN["author_item"])
        if text:
            out.append(text)
        if len(out) >= AUTHOR_MAX_CAPACITY:
            break
    return out

def topic_value(row: dict[str, Any], extra: dict[str, Any], direct_key: str, parent_key: str | None = None, nested_key: str | None = None) -> str:
    direct = first_text(row.get(direct_key), extra.get(direct_key))
    if direct:
        return direct.lower()
    if not parent_key:
        return ""
    parent_raw = row.get(parent_key)
    if parent_raw is None:
        parent_raw = extra.get(parent_key)
    parent_dict = parse_topic_dict(parent_raw)
    if not parent_dict:
        return first_text(parent_raw).lower()
    if nested_key is None:
        return first_text(parent_dict.get("display_name")).lower()
    nested = parent_dict.get(nested_key)
    nested_dict = nested if isinstance(nested, dict) else parse_topic_dict(nested)
    return first_text(nested_dict.get("display_name")).lower()
def build_extra_info(row: dict[str, Any]) -> str:
    extra = parse_extra_info(row.get("extra_info"))
    for key, value in row.items():
        if key.startswith("__"):
            continue
        if key in PROMOTED_EXTRA_INFO_KEYS or key == "embedding":
            continue
        if value is None or value == "":
            continue
        extra[key] = value
    if row.get("title") and "title_original" not in extra:
        extra["title_original"] = row["title"]
    if row.get("author") and "author_original" not in extra:
        extra["author_original"] = row["author"]
    if row.get("publication_venue_name_unified") and "publication_venue_name_unified_original" not in extra:
        extra["publication_venue_name_unified_original"] = row["publication_venue_name_unified"]
    encoded = json.dumps(extra, ensure_ascii=False, separators=(",", ":")) if extra else "{}"
    if len(encoded.encode("utf-8")) <= MAX_LEN["extra_info"]:
        return encoded
    if isinstance(extra.get("abstract"), str):
        extra["abstract"] = truncate_utf8(extra["abstract"], 4096)
        encoded = json.dumps(extra, ensure_ascii=False, separators=(",", ":"))
        if len(encoded.encode("utf-8")) <= MAX_LEN["extra_info"]:
            return encoded
        extra.pop("abstract", None)
        encoded = json.dumps(extra, ensure_ascii=False, separators=(",", ":"))
    return truncate_utf8(encoded, MAX_LEN["extra_info"])
def build_output_row(raw_row: dict[str, Any], vector: list[float]) -> dict[str, Any]:
    extra = parse_extra_info(raw_row.get("extra_info"))
    created_time = normalize_ts(raw_row.get("created_time"))
    updated_time = normalize_ts(raw_row.get("updated_time"), created_time)
    publication_published_date = normalize_published_date(raw_row.get("publication_published_date") or extra.get("publication_published_date"))
    publication_published_year = normalize_int(raw_row.get("publication_published_year") if raw_row.get("publication_published_year") is not None else extra.get("publication_published_year"), 0)
    if publication_published_year <= 0 and publication_published_date > 0:
        publication_published_year = publication_published_date // 10000
    row = {
        "chunk_id": truncate_utf8(raw_row.get("chunk_id"), MAX_LEN["chunk_id"]),
        "doc_id": truncate_utf8(raw_row.get("doc_id"), MAX_LEN["doc_id"]),
        "title": truncate_utf8(first_text(raw_row.get("title")).lower(), MAX_LEN["title"]),
        "url": truncate_utf8(first_text(raw_row.get("url")), MAX_LEN["url"]),
        "embedding": [float(x) for x in vector],
        "offset": max(0, normalize_int(raw_row.get("offset"), 0)),
        "page_no": max(0, normalize_int(raw_row.get("page_no"), 0)),
        "chunk_no": max(0, normalize_int(raw_row.get("chunk_no"), 0)),
        "source_type": truncate_utf8(first_text(raw_row.get("source_type"), extra.get("source_type")), MAX_LEN["source_type"]),
        "lang": truncate_utf8(first_text(raw_row.get("lang"), extra.get("lang"), extra.get("language"), "unknown"), MAX_LEN["lang"]),
        "category": truncate_utf8(first_text(raw_row.get("category")), MAX_LEN["category"]),
        "job_id": truncate_utf8(first_text(raw_row.get("job_id")), MAX_LEN["job_id"]),
        "task_id": truncate_utf8(first_text(raw_row.get("task_id"), str(uuid.uuid4())), MAX_LEN["task_id"]),
        "publication_published_date": publication_published_date,
        "publication_published_year": publication_published_year,
        "metadata_type": truncate_utf8(first_text(raw_row.get("metadata_type"), extra.get("metadata_type")).lower(), MAX_LEN["metadata_type"]),
        "publication_venue_type": truncate_utf8(first_text(raw_row.get("publication_venue_type"), extra.get("publication_venue_type")).lower(), MAX_LEN["publication_venue_type"]),
        "primary_topic": truncate_utf8(topic_value(raw_row, extra, "primary_topic", "primary_topic", None), MAX_LEN["primary_topic"]),
        "primary_topic_subfield": truncate_utf8(topic_value(raw_row, extra, "primary_topic_subfield", "primary_topic", "subfield"), MAX_LEN["primary_topic_subfield"]),
        "primary_topic_field": truncate_utf8(topic_value(raw_row, extra, "primary_topic_field", "primary_topic", "field"), MAX_LEN["primary_topic_field"]),
        "primary_topic_domain": truncate_utf8(topic_value(raw_row, extra, "primary_topic_domain", "primary_topic", "domain"), MAX_LEN["primary_topic_domain"]),
        "author": normalize_author_list(raw_row.get("author") if raw_row.get("author") is not None else extra.get("author"), lowercase=True, debug_context={"chunk_id": raw_row.get("chunk_id"), "doc_id": raw_row.get("doc_id"), "source_file": raw_row.get("__source_file"), "row_loc": raw_row.get("__row_loc")}),
        "publication_venue_name_unified": truncate_utf8(first_text(raw_row.get("publication_venue_name_unified"), extra.get("publication_venue_name_unified")).lower(), MAX_LEN["publication_venue_name_unified"]),
        "access_is_oa": truncate_utf8(normalize_bool_string(raw_row.get("access_is_oa") if raw_row.get("access_is_oa") is not None else extra.get("access_is_oa")), MAX_LEN["access_is_oa"]),
        "citation_count": normalize_int(raw_row.get("citation_count") if raw_row.get("citation_count") is not None else extra.get("citation_count"), -1),
        "influential_citation_count": normalize_int(raw_row.get("influential_citation_count") if raw_row.get("influential_citation_count") is not None else extra.get("influential_citation_count"), -1),
        "extra_info": build_extra_info(raw_row),
        "created_time": created_time,
        "updated_time": updated_time,
    }
    missing = [key for key in ("chunk_id", "doc_id") if not row[key]]
    if missing:
        raise ValueError(f"missing required fields: {missing}")
    return row
def runtime_settings() -> Settings:
    return settings
def build_http_client(runtime_cfg: Settings, max_connections: int = 1) -> httpx.Client:
    connection_count = max(1, int(max_connections))
    return httpx.Client(
        timeout=float(runtime_cfg.embedding_timeout_seconds),
        verify=True,
        limits=httpx.Limits(
            max_connections=connection_count,
            max_keepalive_connections=connection_count,
        ),
    )
def embed_with_retry(records: list[Any], retry_times: int = 3, base_sleep_seconds: float = 2.0, runtime_cfg: Settings | None = None, http: httpx.Client | None = None) -> list[list[float]]:
    last_exc = None
    runtime_cfg = runtime_cfg or runtime_settings()
    owns_http = http is None
    http = http or build_http_client(runtime_cfg)
    try:
        for attempt in range(retry_times + 1):
            try:
                return embed_record_texts(
                    runtime_cfg,
                    http,
                    records,
                    embedding_source="",
                    embedding_model="",
                )
            except Exception as exc:
                last_exc = exc
                if attempt >= retry_times:
                    break
                time.sleep(base_sleep_seconds * (2 ** attempt))
    finally:
        if owns_http:
            http.close()
    raise last_exc
def ensure_s3a(path: str) -> str:
    if path.startswith("s3://"):
        return "s3a://" + path[len("s3://"):]
    return path
def ensure_s3(path: str) -> str:
    if path.startswith("s3a://"):
        return "s3://" + path[len("s3a://"):]
    return path
def is_input_data_file(path: str) -> bool:
    return path.endswith('.jsonl') or path.endswith('.jsonl.gz')
def list_input_files(input_root: str) -> list[str]:
    return sorted(path for path in list_s3_objects(input_root, recursive=True) if is_input_data_file(path))
def load_progress_df(spark, progress_root: str):
    files = sorted(path for path in list_s3_objects(progress_root, recursive=True) if path.endswith('.jsonl'))
    if not files:
        return None
    return spark.read.schema(PROGRESS_SCHEMA).json([ensure_s3a(path) for path in files])
def load_progress_summary(spark, progress_root: str) -> dict[str, Any]:
    progress_df = load_progress_df(spark, progress_root)
    if progress_df is None:
        return {
            'completed_batch_index': 0,
            'completed_row_count': 0,
            'last_completed_source_file': None,
            'last_completed_chunk_id': None,
        }
    latest = progress_df.orderBy(F.col('batch_index').desc()).limit(1).collect()[0]
    return {
        'completed_batch_index': int(latest['batch_index'] or 0),
        'completed_row_count': int(latest['completed_row_count'] or 0),
        'last_completed_source_file': ensure_s3(latest['last_completed_source_file']) if latest['last_completed_source_file'] else None,
        'last_completed_chunk_id': latest['last_completed_chunk_id'],
    }
def write_progress(batch_index: int, batch_start_row: int, batch_end_row: int, completed_row_count: int, last_completed_source_file: str | None, last_completed_chunk_id: str | None, success_row_count: int, error_row_count: int, progress_root: str) -> dict[str, Any]:
    payload = {
        'batch_index': int(batch_index),
        'batch_start_row': int(batch_start_row),
        'batch_end_row': int(batch_end_row),
        'completed_row_count': int(completed_row_count),
        'last_completed_source_file': last_completed_source_file,
        'last_completed_chunk_id': last_completed_chunk_id,
        'success_row_count': int(success_row_count),
        'error_row_count': int(error_row_count),
        'completed_at': utc_now_iso(),
    }
    output_path = f"{progress_root.rstrip('/')}/batch-{int(batch_index):08d}-{uuid.uuid4().hex}.jsonl"
    put_s3_object(output_path, (json.dumps(payload, ensure_ascii=False, separators=(",", ":")) + "\n").encode('utf-8'))
    return payload
def write_batch_json(df, output_root: str, partition_count: int, write_mode: str) -> None:
    payload_json_rdd = df.select('value').rdd.map(lambda row: row['value'])
    (
        spark.read.json(payload_json_rdd)
        .coalesce(max(1, int(partition_count)))
        .write
        .mode(write_mode)
        .json(ensure_s3a(output_root))
    )
def load_summary_payload(summary_path: str) -> dict[str, Any]:
    try:
        return json.loads(read_s3_object_bytes(summary_path).decode('utf-8'))
    except Exception:
        return {}
def count_total_input_rows(input_root: str) -> int:
    payload = load_summary_payload(INPUT_SUMMARY_PATH)
    chunk_count = int(payload.get('chunk_count') or 0)
    if chunk_count <= 0:
        raise RuntimeError(f"input summary missing valid chunk_count: {INPUT_SUMMARY_PATH}")
    return chunk_count

def build_input_candidate_df(file_paths: list[str], last_completed_source_file: str | None, last_completed_chunk_id: str | None):
    candidate_df = (
        spark.read.schema(INPUT_FILE_SCHEMA).json([ensure_s3a(path) for path in file_paths])
        .where(F.col("record_type") == F.lit("chunk"))
        .where(F.col("chunk_id").isNotNull() & (F.trim(F.col("chunk_id")) != ""))
        .withColumn("__source_file", F.regexp_replace(F.input_file_name(), r"^s3a://", "s3://"))
        .withColumn("__row_loc", F.col("chunk_id"))
    )
    if last_completed_source_file:
        candidate_df = candidate_df.where(
            (F.col("__source_file") > F.lit(last_completed_source_file)) |
            ((F.col("__source_file") == F.lit(last_completed_source_file)) & (F.col("chunk_id") > F.lit(last_completed_chunk_id or "")))
        )
    return candidate_df
    
def fetch_row_batch_df(input_files: list[str], last_completed_source_file: str | None, last_completed_chunk_id: str | None, limit_rows: int, avg_rows_per_file: float, file_count_padding_ratio: float = 1.05, full_file_batch_min_ratio: float = 0.98):
    if limit_rows <= 0:
        return None, None, 0, None, None, 0
    start_index = bisect_left(input_files, last_completed_source_file) if last_completed_source_file else 0
    if start_index >= len(input_files):
        return None, None, 0, None, None, 0
    file_count = max(1, int(math.ceil((limit_rows / max(1.0, avg_rows_per_file)) * max(1.0, file_count_padding_ratio))))
    file_count = min(len(input_files) - start_index, file_count)
    scanned_end_index = start_index
    positive_stats = []
    while start_index < len(input_files):
        target_end_index = min(len(input_files), start_index + file_count)
        delta_file_paths = input_files[scanned_end_index: target_end_index]
        if not delta_file_paths and scanned_end_index >= len(input_files):
            return None, None, 0, None, None, 0
        if delta_file_paths:
            delta_df = build_input_candidate_df(delta_file_paths, last_completed_source_file, last_completed_chunk_id)
            delta_file_stats_rows = delta_df.groupBy("__source_file").agg(
                F.count(F.lit(1)).alias("row_count"),
                F.max(F.col("chunk_id")).alias("max_chunk_id"),
            ).collect()
            delta_stats_by_file = {row["__source_file"]: row for row in delta_file_stats_rows}
            for path in delta_file_paths:
                row = delta_stats_by_file.get(path)
                if row is None or int(row["row_count"] or 0) <= 0:
                    continue
                positive_stats.append(row)
        if not positive_stats:
            if target_end_index >= len(input_files):
                return None, None, 0, None, None, 0
            scanned_end_index = target_end_index
            file_count = min(len(input_files) - start_index, max(file_count + 1, file_count * 2))
            continue
        selected_files = []
        selected_row_count = 0
        batch_last_source_file = None
        batch_last_chunk_id = None
        batch_df = None
        batch_cache_df = None
        batch_file_count = 0
        batch_is_finalized = False
        for row in positive_stats:
            source_file = row["__source_file"]
            row_count = int(row["row_count"] or 0)
            max_chunk_id = row["max_chunk_id"]
            projected_row_count = selected_row_count + row_count
            if projected_row_count < limit_rows:
                selected_files.append(source_file)
                selected_row_count = projected_row_count
                batch_last_source_file = source_file
                batch_last_chunk_id = max_chunk_id
                continue
            if selected_row_count > 0 and selected_row_count >= int(math.ceil(limit_rows * full_file_batch_min_ratio)):
                batch_cache_df = build_input_candidate_df(selected_files, last_completed_source_file, last_completed_chunk_id).persist(StorageLevel.MEMORY_AND_DISK)
                batch_df = batch_cache_df
                batch_file_count = len(selected_files)
                batch_is_finalized = True
                break
            tail_limit_rows = max(1, limit_rows - selected_row_count)
            tail_file_paths = selected_files + [source_file]
            batch_cache_df = build_input_candidate_df(tail_file_paths, last_completed_source_file, last_completed_chunk_id).persist(StorageLevel.MEMORY_AND_DISK)
            tail_df = batch_cache_df.where(F.col("__source_file") == F.lit(source_file)).orderBy(F.col("chunk_id").asc()).limit(tail_limit_rows)
            tail_row = tail_df.select(
                F.count(F.lit(1)).alias("row_count"),
                F.max(F.col("chunk_id")).alias("last_chunk_id"),
            ).collect()[0]
            tail_count = int(tail_row["row_count"] or 0)
            if selected_files:
                batch_df = batch_cache_df.where(F.col("__source_file").isin(selected_files)).unionByName(tail_df)
            else:
                batch_df = tail_df
            selected_row_count += tail_count
            batch_last_source_file = source_file if tail_count > 0 else batch_last_source_file
            batch_last_chunk_id = tail_row["last_chunk_id"] if tail_count > 0 else batch_last_chunk_id
            batch_file_count = len(selected_files) + (1 if tail_count > 0 else 0)
            batch_is_finalized = True
            break
        if not batch_is_finalized:
            batch_file_paths = [row["__source_file"] for row in positive_stats]
            batch_cache_df = build_input_candidate_df(batch_file_paths, last_completed_source_file, last_completed_chunk_id).persist(StorageLevel.MEMORY_AND_DISK)
            batch_df = batch_cache_df
            batch_file_count = len(positive_stats)
            last_row = positive_stats[-1]
            batch_last_source_file = last_row["__source_file"]
            batch_last_chunk_id = last_row["max_chunk_id"]
            batch_is_finalized = selected_row_count >= limit_rows or target_end_index >= len(input_files)
        if batch_is_finalized and selected_row_count > 0:
            return batch_df, batch_cache_df, selected_row_count, batch_last_source_file, batch_last_chunk_id, batch_file_count
        if batch_cache_df is not None:
            batch_cache_df.unpersist()
        if target_end_index >= len(input_files):
            return None, None, 0, None, None, 0
        scanned_end_index = target_end_index
        file_count = min(len(input_files) - start_index, max(file_count + 1, file_count * 2))
    return None, None, 0, None, None, 0
    
def build_summary_payload(job, total_input_rows: int, completed_row_count: int, completed_batch_index: int, last_completed_source_file: str | None, last_completed_chunk_id: str | None, success_row_count_total: int, error_row_count_total: int, previous_summary: dict[str, Any] | None = None) -> dict[str, Any]:
    pending_row_count = max(0, total_input_rows - completed_row_count)
    total_batch_count = math.ceil(total_input_rows / max(1, job.batch_row_count)) if total_input_rows > 0 else 0
    previous_summary = previous_summary or {}
    previous_success_rows = int(previous_summary.get('success_rows') or 0)
    previous_error_rows = int(previous_summary.get('error_rows') or 0)
    previous_processed_rows = int(previous_summary.get('processed_rows') or 0)
    processed_rows = max(int(success_row_count_total + error_row_count_total), previous_processed_rows + int(success_row_count_total + error_row_count_total))
    success_rows = previous_success_rows + int(success_row_count_total)
    error_rows = previous_error_rows + int(error_row_count_total)
    return {
        'input_root': job.input_root,
        'output_root': job.output_root,
        'progress_root': job.progress_root,
        'total_input_rows': int(total_input_rows),
        'completed_row_count': int(completed_row_count),
        'pending_row_count': int(pending_row_count),
        'batch_row_count': int(job.batch_row_count),
        'completed_batch_index': int(completed_batch_index),
        'executed_batch_count': int(completed_batch_index),
        'total_batch_count': int(total_batch_count),
        'last_completed_source_file': last_completed_source_file,
        'last_completed_chunk_id': last_completed_chunk_id,
        'processed_rows': int(processed_rows),
        'success_rows': int(success_rows),
        'error_rows': int(error_rows),
        'success_row_count_total': int(success_rows),
        'error_row_count_total': int(error_rows),
        'milvus_fields': MILVUS_FIELDS,
        'scalar_index_fields': SCALAR_INDEX_FIELDS,
        'updated_at': utc_now_iso(),
    }

@dataclass
class JobConfig:
    output_root: str
    progress_root: str
    summary_path: str
    error_root: str
    input_root: str = INPUT_ROOT
    batch_row_count: int = 5000
    embed_batch_size: int = max(1, int(settings.embedding_max_batch or 32))
    process_partitions: int = 64
    output_partitions: int = 64
    partition_embed_concurrency: int = 1
    partition_http_connections: int | None = None
    fetch_file_count_padding_ratio: float = 1.05
    full_file_batch_min_ratio: float = 0.98
    partition_target_rows_multiplier: float = 1.5
    shuffle_repartition_ratio: float | None = None
    test_limit_rows: int | None = None
    overwrite_output: bool = OVERWRITE_OUTPUT
    fail_fast: bool = True
    
def run_job(job: JobConfig) -> dict[str, Any]:
    job_started_at = time.perf_counter()
    if job.overwrite_output:
        stage_started_at = time.perf_counter()
        delete_s3_objects(job.output_root, dry_run=False, verbose=True)
        log_info(f"清理输出目录耗时={elapsed_seconds(stage_started_at):.2f}s")
    stage_started_at = time.perf_counter()
    input_files = list_input_files(job.input_root)
    log_info(f"列出输入文件耗时={elapsed_seconds(stage_started_at):.2f}s，文件数={len(input_files)}")
    stage_started_at = time.perf_counter()
    progress_summary = load_progress_summary(spark, job.progress_root)
    log_info(f"读取续跑进度耗时={elapsed_seconds(stage_started_at):.2f}s")
    completed_batch_index = int(progress_summary['completed_batch_index'])
    completed_row_count = int(progress_summary['completed_row_count'])
    last_completed_source_file = progress_summary['last_completed_source_file']
    last_completed_chunk_id = progress_summary['last_completed_chunk_id']
    stage_started_at = time.perf_counter()
    summary_payload = load_summary_payload(job.summary_path)
    log_info(f"读取输出 summary 耗时={elapsed_seconds(stage_started_at):.2f}s")
    input_summary_payload = load_summary_payload(INPUT_SUMMARY_PATH)
    log_info(f"读取输入 summary 耗时={elapsed_seconds(stage_started_at):.2f}s")
    total_input_rows = int(input_summary_payload.get('chunk_count') or 0)
    if total_input_rows <= 0:
        stage_started_at = time.perf_counter()
        total_input_rows = count_total_input_rows(job.input_root)
        log_info(f"读取输入源总行数耗时={elapsed_seconds(stage_started_at):.2f}s，总行数={total_input_rows}")
    pending_row_count = max(0, total_input_rows - completed_row_count)
    if job.test_limit_rows is not None:
        pending_row_count = min(pending_row_count, int(job.test_limit_rows))
    pending_batch_count = math.ceil(pending_row_count / max(1, job.batch_row_count)) if pending_row_count > 0 else 0
    total_batch_count = completed_batch_index + pending_batch_count
    log_info(
        f"总行数={total_input_rows}，已完成行数={completed_row_count}，剩余行数={pending_row_count}，"
        f"批大小={job.batch_row_count}，总批次={total_batch_count}"
    )
    global_batch_index = completed_batch_index
    cursor_source_file = last_completed_source_file
    cursor_chunk_id = last_completed_chunk_id
    success_row_count_total = 0
    error_row_count_total = 0
    remaining_rows = pending_row_count
    base_completed_row_count = completed_row_count
    avg_rows_per_file = max(1.0, float(total_input_rows) / max(1, len(input_files)))
    while remaining_rows > 0:
        current_batch_size = min(job.batch_row_count, remaining_rows)
        stage_started_at = time.perf_counter()
        batch_df, batch_cache_df, batch_row_count, batch_last_source_file, batch_last_chunk_id, batch_file_count = fetch_row_batch_df(
            input_files,
            cursor_source_file,
            cursor_chunk_id,
            current_batch_size,
            avg_rows_per_file,
            file_count_padding_ratio=job.fetch_file_count_padding_ratio,
            full_file_batch_min_ratio=job.full_file_batch_min_ratio,
        )
        log_info(f"抓取批次数据耗时={elapsed_seconds(stage_started_at):.2f}s，请求条数={current_batch_size}，实际条数={batch_row_count}，文件数={batch_file_count}")
        if batch_row_count <= 0:
            if batch_cache_df is not None:
                batch_cache_df.unpersist()
            break
        if batch_file_count > 0:
            avg_rows_per_file = max(1.0, float(batch_row_count) / max(1, batch_file_count))
        global_batch_index += 1
        batch_start_row = completed_row_count + 1
        batch_end_row = completed_row_count + batch_row_count
        partition_target_rows = max(
            1,
            int(math.ceil(job.embed_batch_size * max(1, job.partition_embed_concurrency) * max(1.0, job.partition_target_rows_multiplier))),
        )
        target_process_partitions = max(1, min(job.process_partitions, math.ceil(batch_row_count / partition_target_rows)))
        input_partition_count = batch_df.rdd.getNumPartitions()
        if job.shuffle_repartition_ratio is not None and input_partition_count >= int(math.ceil(target_process_partitions * max(1.0, job.shuffle_repartition_ratio))):
            process_df = batch_df.repartition(target_process_partitions)
            process_partition_strategy = 'shuffle_repartition'
            effective_process_partitions = target_process_partitions
        elif input_partition_count > target_process_partitions:
            process_df = batch_df.coalesce(target_process_partitions)
            process_partition_strategy = 'coalesce'
            effective_process_partitions = target_process_partitions
        elif input_partition_count < max(1, target_process_partitions // 2):
            process_df = batch_df.repartition(target_process_partitions)
            process_partition_strategy = 'shuffle_repartition'
            effective_process_partitions = target_process_partitions
        else:
            process_df = batch_df
            process_partition_strategy = 'reuse_input'
            effective_process_partitions = input_partition_count
        effective_http_connections = max(1, int(job.partition_http_connections or job.partition_embed_concurrency))
        max_buffer_size = max(1, job.embed_batch_size * max(1, job.partition_embed_concurrency))
        log_info(
            f"批次 {global_batch_index}/{total_batch_count}: rows={batch_start_row}-{batch_end_row}/{base_completed_row_count + pending_row_count}，"
            f"本批={batch_row_count}，文件数={batch_file_count}，分区={effective_process_partitions}(目标={target_process_partitions}，输入={input_partition_count}，策略={process_partition_strategy}，目标行/分区={partition_target_rows})，HTTP连接={effective_http_connections}"
        )
        batch_started_at = time.perf_counter()
        class EmbedTextRecord:
            __slots__ = ("text",)
            def __init__(self, text: str):
                self.text = text
        def process_partition(rows):
            buffer = []
            partition_metric_calls = 0
            partition_metric_texts = 0
            partition_metric_seconds = 0.0
            runtime_cfg = runtime_settings()
            metric_lock = threading.Lock()
            client_lock = threading.Lock()
            thread_local = threading.local()
            shared_http = build_http_client(runtime_cfg, max_connections=effective_http_connections) if job.partition_embed_concurrency <= 1 else None
            partition_http_clients = []
            pool = ThreadPoolExecutor(max_workers=job.partition_embed_concurrency) if job.partition_embed_concurrency > 1 else None
            def get_http_client():
                if pool is None:
                    return shared_http
                http = getattr(thread_local, 'http', None)
                if http is None:
                    http = build_http_client(runtime_cfg, max_connections=1)
                    thread_local.http = http
                    with client_lock:
                        partition_http_clients.append(http)
                return http
            def make_error_row(item, error_message, traceback_text):
                err = {
                    'record_type': 'error',
                    'batch_index': global_batch_index,
                    'chunk_id': item.get('chunk_id'),
                    'doc_id': item.get('doc_id'),
                    'row_loc': item.get('__row_loc'),
                    'error_message': error_message,
                    'traceback': traceback_text,
                }
                return Row(value=json.dumps(err, ensure_ascii=False), sub_path='error', metric_calls=0, metric_texts=0, metric_seconds=0.0)
            def make_success_row(item, vector):
                out = build_output_row(item, vector)
                out['sub_path'] = f"source_type={out['source_type'] or 'unknown'}/lang={out['lang'] or 'unknown'}"
                return Row(value=json.dumps(out, ensure_ascii=False), sub_path=out['sub_path'], metric_calls=0, metric_texts=0, metric_seconds=0.0)
            def flush_buffer():
                nonlocal buffer, partition_metric_calls, partition_metric_texts, partition_metric_seconds
                if not buffer:
                    return
                def embed_records(records):
                    nonlocal partition_metric_calls, partition_metric_texts, partition_metric_seconds
                    started_at = time.perf_counter()
                    vectors = embed_with_retry(records, runtime_cfg=runtime_cfg, http=get_http_client())
                    with metric_lock:
                        partition_metric_calls += 1
                        partition_metric_texts += len(records)
                        partition_metric_seconds += elapsed_seconds(started_at)
                    return vectors
                def diagnose_failed_items(items, batch_exc):
                    failed_items = []
                    for item in items:
                        try:
                            vectors = embed_records([EmbedTextRecord(item['content'])])
                            if len(vectors) != 1:
                                raise ValueError(f"embedding result size mismatch for single item: outputs={len(vectors)}")
                        except Exception as single_exc:
                            failed_item = build_debug_item_payload(item)
                            failed_item["embedding_error"] = extract_embedding_error_response(single_exc)
                            failed_items.append(failed_item)
                    return build_embed_batch_error_message(batch_exc, global_batch_index, items, failed_items), failed_items
                def emit_chunk_results(items_chunk, vectors):
                    for item, vector in zip(items_chunk, vectors):
                        try:
                            yield make_success_row(item, vector)
                        except Exception as exc:
                            yield make_error_row(item, str(exc), traceback.format_exc())
                            if job.fail_fast:
                                raise
                try:
                    if job.partition_embed_concurrency <= 1 or len(buffer) <= job.embed_batch_size:
                        vectors = embed_records([EmbedTextRecord(item['content']) for item in buffer])
                        yield from emit_chunk_results(buffer, vectors)
                    else:
                        chunk_size = max(1, job.embed_batch_size)
                        chunks = [buffer[idx: idx + chunk_size] for idx in range(0, len(buffer), chunk_size)]
                        future_map = {
                            pool.submit(embed_records, [EmbedTextRecord(item['content']) for item in chunk]): chunk
                            for chunk in chunks
                        }
                        for future in as_completed(future_map):
                            chunk = future_map[future]
                            yield from emit_chunk_results(chunk, future.result())
                except Exception as exc:
                    debug_message, failed_items = diagnose_failed_items(buffer, exc)
                    if failed_items and all(is_context_length_error(item.get('embedding_error') or {}) for item in failed_items):
                        failed_item_map = {(item.get('chunk_id'), item.get('row_loc')): item for item in failed_items}
                        for item in buffer:
                            key = (item.get('chunk_id'), item.get('__row_loc'))
                            failed_item = failed_item_map.get(key)
                            if failed_item is not None:
                                yield make_error_row(item, json.dumps(failed_item, ensure_ascii=False), '')
                                continue
                            try:
                                vectors = embed_records([EmbedTextRecord(item['content'])])
                                if len(vectors) != 1:
                                    raise ValueError(f"embedding result size mismatch for single item recovery: outputs={len(vectors)}")
                                yield make_success_row(item, vectors[0])
                            except Exception as recover_exc:
                                yield make_error_row(item, str(recover_exc), traceback.format_exc())
                                if job.fail_fast:
                                    raise
                    else:
                        if job.fail_fast:
                            raise RuntimeError(debug_message) from exc
                        for item in buffer:
                            yield make_error_row(item, debug_message, traceback.format_exc())
                finally:
                    buffer = []
            try:
                for item in rows:
                    row = item.asDict(recursive=True) if hasattr(item, 'asDict') else dict(item)
                    content = first_text(row.get('content'))
                    if not content:
                        yield make_error_row(row, 'content is empty', '')
                        continue
                    row['content'] = content
                    buffer.append(row)
                    if len(buffer) >= max_buffer_size:
                        yield from flush_buffer()
                yield from flush_buffer()
                yield Row(value='{}', sub_path='__metric__', metric_calls=partition_metric_calls, metric_texts=partition_metric_texts, metric_seconds=partition_metric_seconds)
            finally:
                if pool is not None:
                    pool.shutdown(wait=True)
                if shared_http is not None:
                    shared_http.close()
                for http in partition_http_clients:
                    http.close()
        result_rdd = process_df.rdd.mapPartitions(process_partition)
        stage_started_at = time.perf_counter()
        result_df = spark.createDataFrame(result_rdd, schema=RESULT_ROW_SCHEMA).persist(StorageLevel.MEMORY_AND_DISK)
        metrics_started_at = time.perf_counter()
        metrics_row = result_df.where(F.col('sub_path') == F.lit('__metric__')).select(
            F.sum(F.col('metric_calls')).alias('embed_calls'),
            F.sum(F.col('metric_texts')).alias('embed_texts'),
            F.sum(F.col('metric_seconds')).alias('embed_seconds'),
        ).collect()[0]
        embed_calls = int(metrics_row['embed_calls'] or 0)
        embed_texts = int(metrics_row['embed_texts'] or 0)
        embed_seconds = float(metrics_row['embed_seconds'] or 0.0)
        embed_wall_seconds = elapsed_seconds(metrics_started_at)
        avg_call_seconds = (embed_seconds / embed_calls) if embed_calls > 0 else 0.0
        equivalent_parallelism = (embed_seconds / embed_wall_seconds) if embed_wall_seconds > 0 else 0.0
        theoretical_min_calls = max(1, int(math.ceil(embed_texts / max(1, job.embed_batch_size)))) if embed_texts > 0 else 0
        extra_calls = max(0, embed_calls - theoretical_min_calls)
        log_info(
            f"embedding调用汇总耗时={embed_wall_seconds:.2f}s，calls={embed_calls}，texts={embed_texts}，"
            f"executor_embed_seconds={embed_seconds:.2f}s，avg_call_seconds={avg_call_seconds:.2f}s，并行折算={equivalent_parallelism:.1f}，"
            f"理论最少calls={theoretical_min_calls}，额外calls={extra_calls}"
        )
        stage_started_at = time.perf_counter()
        payload_df = result_df.where(F.col('sub_path') != F.lit('__metric__')).persist(StorageLevel.MEMORY_AND_DISK)
        counts_row = payload_df.select(
            F.sum(F.when(F.col('sub_path') == F.lit('error'), F.lit(1)).otherwise(F.lit(0))).alias('error_row_count'),
            F.sum(F.when(F.col('sub_path') != F.lit('error'), F.lit(1)).otherwise(F.lit(0))).alias('success_row_count'),
            F.count(F.lit(1)).alias('materialized_row_count'),
        ).collect()[0]
        success_row_count = int(counts_row['success_row_count'] or 0)
        error_row_count = int(counts_row['error_row_count'] or 0)
        materialized_row_count = int(counts_row['materialized_row_count'] or 0)
        log_info(f"payload物化+统计耗时={elapsed_seconds(stage_started_at):.2f}s，结果行数={materialized_row_count}，success={success_row_count}，error={error_row_count}")
        success_df = payload_df.where(F.col('sub_path') != F.lit('error'))
        error_df = payload_df.where(F.col('sub_path') == F.lit('error'))
        batch_output_root = f"{job.output_root.rstrip('/')}/data/batch_id={global_batch_index:08d}"
        batch_error_root = f"{job.error_root.rstrip('/')}/batch_id={global_batch_index:08d}"
        if DELETE_BATCH_OUTPUT_BEFORE_WRITE and (success_row_count > 0 or error_row_count > 0):
            stage_started_at = time.perf_counter()
            delete_s3_objects(batch_output_root, dry_run=False, verbose=False)
            delete_s3_objects(batch_error_root, dry_run=False, verbose=False)
            log_info(f"清理批次历史输出耗时={elapsed_seconds(stage_started_at):.2f}s")
        if success_row_count > 0 and OUTPUT_WRITE_MODE == 'append' and path_exists(batch_output_root.replace('s3://', 's3a://', 1)):
            raise RuntimeError(f"append 模式下检测到已存在的批次输出目录，说明上次可能已写出但未更新 progress: {batch_output_root}")
        if error_row_count > 0 and OUTPUT_WRITE_MODE == 'append' and path_exists(batch_error_root.replace('s3://', 's3a://', 1)):
            raise RuntimeError(f"append 模式下检测到已存在的错误批次输出目录，说明上次可能已写出但未更新 progress: {batch_error_root}")
        if success_row_count > 0:
            stage_started_at = time.perf_counter()
            write_batch_json(success_df, batch_output_root, job.output_partitions, OUTPUT_WRITE_MODE)
            log_info(f"写出成功结果耗时={elapsed_seconds(stage_started_at):.2f}s")
        if error_row_count > 0:
            stage_started_at = time.perf_counter()
            write_batch_json(error_df, batch_error_root, max(1, min(job.output_partitions, 32)), OUTPUT_WRITE_MODE)
            log_info(f"写出错误结果耗时={elapsed_seconds(stage_started_at):.2f}s")
        payload_df.unpersist()
        result_df.unpersist()
        completed_row_count += materialized_row_count
        cursor_source_file = batch_last_source_file
        cursor_chunk_id = batch_last_chunk_id
        if batch_cache_df is not None:
            batch_cache_df.unpersist()
        success_row_count_total += success_row_count
        error_row_count_total += error_row_count
        remaining_rows -= batch_row_count
        stage_started_at = time.perf_counter()
        write_progress(
            batch_index=global_batch_index,
            batch_start_row=batch_start_row,
            batch_end_row=batch_end_row,
            completed_row_count=completed_row_count,
            last_completed_source_file=cursor_source_file,
            last_completed_chunk_id=cursor_chunk_id,
            success_row_count=success_row_count,
            error_row_count=error_row_count,
            progress_root=job.progress_root,
        )
        log_info(f"写入进度耗时={elapsed_seconds(stage_started_at):.2f}s")
        log_info(
            f"批次 {global_batch_index}/{total_batch_count} 完成：success={success_row_count}，error={error_row_count}，"
            f"耗时={elapsed_seconds(batch_started_at):.2f}s"
        )
        if job.test_limit_rows is not None and remaining_rows <= 0:
            break
    stage_started_at = time.perf_counter()
    final_summary = build_summary_payload(
        job=job,
        total_input_rows=total_input_rows,
        completed_row_count=completed_row_count,
        completed_batch_index=global_batch_index,
        last_completed_source_file=cursor_source_file,
        last_completed_chunk_id=cursor_chunk_id,
        success_row_count_total=success_row_count_total,
        error_row_count_total=error_row_count_total,
        previous_summary=summary_payload,
    )
    put_s3_object(job.summary_path, json.dumps(final_summary, ensure_ascii=False, indent=2).encode('utf-8'))
    log_info(f"写入最终 summary 耗时={elapsed_seconds(stage_started_at):.2f}s")
    log_info(
        f"任务完成：累计完成={completed_row_count}/{total_input_rows}，累计批次={global_batch_index}/{final_summary['total_batch_count']}，"
        f"总耗时={elapsed_seconds(job_started_at):.2f}s"
    )
    return final_summary



patched xinghe s3 multipart upload checksum=SHA256


# main process

In [5]:
# ===== 测试：20 条数据 =====
if spark_profile == "prod":
    print("spark_profile=prod，跳过测试单元格；如需执行请切回 test 或运行全量单元格")
    # ===== 全量运行 =====
    # prod 环境请确认输出目录与 .env 配置后再执行
    # 说明：
    # - 续跑依据：`_progress/batch_id=xxxxxx.jsonl`
    # - 已完成输入文件会自动跳过
    # - 输出按 `data/batch_id=xxxxxx/*.json` 批次写出
    
    prod_job = JobConfig(
        output_root=OUTPUT_ROOT,
        progress_root=PROGRESS_ROOT,
        summary_path=SUMMARY_PATH,
        error_root=ERROR_ROOT,
        batch_row_count=800000,
        embed_batch_size=32,
        process_partitions=384,
        output_partitions=256,
        partition_embed_concurrency=1,
        partition_http_connections=2,
        partition_target_rows_multiplier=1.5,
        test_limit_rows=None,
        overwrite_output=OVERWRITE_OUTPUT,
        fail_fast=True,
    )
    prod_summary = run_job(prod_job)
    prod_summary
    
else:
    test_job = JobConfig(
        output_root=TEST_OUTPUT_ROOT,
        progress_root=TEST_PROGRESS_ROOT,
        summary_path=TEST_SUMMARY_PATH,
        error_root=TEST_ERROR_ROOT,
        batch_row_count=20,
        embed_batch_size=20,
        output_partitions=8,
        test_limit_rows=20,
        overwrite_output=OVERWRITE_OUTPUT,
        fail_fast=True,
    )
    test_summary = run_job(test_job)
    assert test_summary['processed_rows'] == 20, test_summary
    assert test_summary['success_rows'] == 20, test_summary
    assert test_summary['error_rows'] == 0, test_summary
    assert test_summary['executed_batch_count'] >= 1, test_summary
    assert len(test_summary['milvus_fields']) == 29, test_summary
    assert set(test_summary['scalar_index_fields']).issubset(set(test_summary['milvus_fields']))
    test_summary


列出输入文件耗时=1.20s，文件数=40
读取续跑进度耗时=0.03s
读取输出 summary 耗时=0.03s
读取输入源总行数耗时=5.05s，总行数=2302
总行数=2302，已完成行数=0，剩余行数=20，批大小=20，总批次=1


[Stage 3:>                                                          (0 + 1) / 1]

抓取批次数据耗时=9.13s，请求条数=20，实际条数=20，文件数=1
批次 1/1: rows=1-20/20，本批=20，文件数=1，分区=1(目标=1，输入=1，策略=reuse_input，目标行/分区=30)，HTTP连接=1


26/06/25 15:51:07 ERROR TaskSetManager: Task 0 in stage 5.0 failed 4 times; aborting job


Py4JJavaError: An error occurred while calling o255.collectToPython.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 5.0 failed 4 times, most recent failure: Lost task 0.3 in stage 5.0 (TID 7) (10.112.31.218 executor 2): org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/tmp/ipykernel_292071/1942030858.py", line 723, in flush_buffer
  File "/tmp/ipykernel_292071/1942030858.py", line 715, in emit_chunk_results
  File "/tmp/ipykernel_292071/1942030858.py", line 684, in make_success_row
  File "/tmp/ipykernel_292071/1942030858.py", line 286, in build_output_row
  File "/tmp/ipykernel_292071/1942030858.py", line 206, in normalize_author_list
AttributeError: 'dict' object has no attribute 'lower'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/opt/spark/python/lib/pyspark.zip/pyspark/worker.py", line 1247, in main
    process()
  File "/opt/spark/pyth